# Chapter 4 · Time Series with Linear Models

Turn a recorded signal into a table, fit a simple model on its own past, and test honestly whether it can forecast.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-04-time-series.ipynb)

- Lecture: [en-04-time-series.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-04-time-series.md)
- Lab: [LM401](https://rathachai.github.io/DA-LAB/learnings/linear/lm401.html)

Three short extras follow: (A) multi-step forecast, (B) window size, (C) random vs chronological split.

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---

## Step 1 · Load libraries
pandas holds tables, numpy does maths, scikit-learn has the model and the scores, matplotlib only draws. Run the cells from top to bottom.

pandas เก็บตาราง, numpy คำนวณทางคณิตศาสตร์, scikit-learn มีโมเดลและคะแนนประเมิน, matplotlib ใช้วาดกราฟเท่านั้น ให้รันเซลล์จากบนลงล่าง

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)

plt.rcParams["figure.figsize"] = (9, 4)

print("Libraries ready")

## Step 2 · Get data
`lm401_timeseries.csv` has a time index `t` and ten 200-point signals (like channels from a data logger). We read it straight from the web.

`lm401_timeseries.csv` มีดัชนีเวลา `t` และสัญญาณ 10 สัญญาณ สัญญาณละ 200 จุด (เหมือนช่องสัญญาณจากเครื่องบันทึกข้อมูล) เราอ่านจากเว็บโดยตรง

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_timeseries.csv"
df = pd.read_csv(url)
print("Columns:", list(df.columns))

## Step 3 · Display data
Look at the first rows, the table size, and a summary (typical level = mean, spread = std, smallest and largest value).

ดูแถวแรก ๆ ขนาดตาราง และสรุปสถิติ (ระดับทั่วไป = ค่าเฉลี่ย, การกระจาย = std, ค่าน้อยสุดและมากสุด)

In [ ]:
print("shape (rows, columns):", df.shape)

In [ ]:
df

In [ ]:
df.describe().round(2)

## Step 4 · Visualize correlation
First, look at the signals: trend, repeating season, a random walk that wanders, and white noise with no pattern. Then ask: *is today's value related to yesterday's?* If yes, the past can predict the future.

ก่อนอื่นให้ดูสัญญาณ: แนวโน้ม (trend), ฤดูกาลที่ซ้ำเป็นรอบ (season), การเดินสุ่ม (random walk) ที่เคลื่อนไปเรื่อย ๆ และสัญญาณรบกวนขาว (white noise) ที่ไม่มีรูปแบบ จากนั้นถามว่า: *ค่าวันนี้สัมพันธ์กับค่าเมื่อวานหรือไม่?* ถ้าใช่ อดีตก็ใช้ทำนายอนาคตได้

In [ ]:
names = [col for col in df.columns if col != "t"]
fig, axes = plt.subplots(5, 2, figsize=(11, 10), sharex=True)
for ax, name in zip(axes.ravel(), names):
    ax.plot(df["t"], df[name], lw=1.2)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
SELECTED_TIME_SERIES_SIGNAL = "trend_season"   # change the column name here, everything below follows

df[SELECTED_TIME_SERIES_SIGNAL].plot(figsize=(9, 3))
plt.title(f"{SELECTED_TIME_SERIES_SIGNAL} over time"); plt.xlabel("time index t"); plt.ylabel("value"); plt.show()

In [ ]:
s = df[SELECTED_TIME_SERIES_SIGNAL]
r = s.corr(s.shift(1))                      # correlation of y(t) with y(t-1)
plt.scatter(s.shift(1), s, s=12)
plt.title(f"{SELECTED_TIME_SERIES_SIGNAL}: y(t) vs y(t-1), r = {r:.3f}")
plt.xlabel("y(t-1)  (yesterday)"); plt.ylabel("y(t)  (today)"); plt.show()

Autocorrelation = the same correlation for several lags (how many steps back). High bars mean the past is useful. Compare a trending season with a random walk and pure noise.

สหสัมพันธ์ในตัวเอง (Autocorrelation) คือสหสัมพันธ์แบบเดียวกันที่คำนวณหลายค่าหน่วงเวลา (lag: ย้อนหลังกี่ก้าว) แท่งสูงหมายถึงอดีตมีประโยชน์ ให้เปรียบเทียบฤดูกาลที่มีแนวโน้ม กับการเดินสุ่ม และสัญญาณรบกวนล้วน

In [ ]:
lags = range(1, 7)
ac = pd.DataFrame({n: [df[n].corr(df[n].shift(k)) for k in lags]
                   for n in [SELECTED_TIME_SERIES_SIGNAL, "random_walk", "white_noise"]}, index=lags)
ac.index.name = "lag"
display(ac.round(3))

In [ ]:
ac.plot.bar(rot=0); plt.title("Autocorrelation by lag"); plt.ylabel("r"); plt.show()

## Step 5 · Process data
A series is **one array** `y0, y1, ..., yn` (`n+1` values) in time order. There is no input column and no answer column, so we build them with a **sliding window** of size `w`: the inputs are the previous `w` values (`x1` oldest ... `xw` newest) and the answer `y` is the value that comes next. In pandas, `shift(k)` slides a column down by `k` rows, which gives us the past. The first `w` rows have no full window and are dropped, so we get **`n+1-w` rows**. Here `w = 5` on `trend_season`.

อนุกรมเวลาคือ **อาร์เรย์เดียว** `y0, y1, ..., yn` (`n+1` ค่า) เรียงตามเวลา ไม่มีคอลัมน์อินพุตและคอลัมน์คำตอบ เราจึงสร้างขึ้นด้วย **หน้าต่างเลื่อน (sliding window)** ขนาด `w`: อินพุตคือ `w` ค่าก่อนหน้า (`x1` เก่าสุด ... `xw` ใหม่สุด) และคำตอบ `y` คือค่าถัดไป ใน pandas `shift(k)` เลื่อนคอลัมน์ลง `k` แถว ซึ่งให้ค่าในอดีตแก่เรา `w` แถวแรกมีหน้าต่างไม่ครบจึงถูกตัดทิ้ง เราจึงได้ **`n+1-w` แถว** ในที่นี้ `w = 5` กับ `trend_season`

In [ ]:
w = 5
s = df[SELECTED_TIME_SERIES_SIGNAL]
df_w = pd.DataFrame({f"x{k}": s.shift(w - k + 1) for k in range(1, w + 1)})  # x1 = shift(5) ... x5 = shift(1)
df_w["y"] = s                       # the value that comes next
df_w = df_w.dropna()
df_w.index.name = "t"               # t = time index of the target
df_w

In [ ]:
print("df rows n+1 =", len(df), "| window w =", w, "| df_w rows n+1-w =", len(df_w))

In [ ]:
print("df_w shape:", df_w.shape, "(w inputs + 1 target)")

Check against the ready-made table the course ships (`lm401_window_w5_trend_season.csv`).

ตรวจสอบเทียบกับตารางสำเร็จรูปที่ใช้ในรายวิชา (`lm401_window_w5_trend_season.csv`)

In [ ]:
url_w = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm401_window_w5_trend_season.csv"
ready = pd.read_csv(url_w).set_index("t")
cols = [f"x{k}" for k in range(1, w + 1)] + ["y"]
print("same shape:", ready[cols].shape == df_w.shape)

In [ ]:
print("same values:", np.allclose(df_w[cols].to_numpy(), ready[cols].to_numpy()))

## Step 6 · Select X and y
`X` = the inputs (the last 5 values), `y` = the answer (the next value).

`X` = อินพุต (5 ค่าล่าสุด), `y` = คำตอบ (ค่าถัดไป)

In [ ]:
feats = [f"x{k}" for k in range(1, w + 1)]
X = df_w[feats]
y = df_w["y"]
print("X:", X.shape, " y:", y.shape)

In [ ]:
X

## Step 7 · Split into train and test
We train on the **early** 70 % and test on the **late** 30 %. We do not shuffle: real forecasting predicts the future from the past, and neighbouring rows share most of their numbers, so a random split would let the model peek at near-copies of the test rows (leakage; see Section C).

ฝึกกับ **ช่วงแรก** 70 % และทดสอบกับ **ช่วงหลัง** 30 % เราไม่สับข้อมูล: การพยากรณ์จริงคือการทำนายอนาคตจากอดีต และแถวที่อยู่ติดกันมีตัวเลขซ้ำกันเกือบหมด การแบ่งแบบสุ่มจึงทำให้โมเดลแอบเห็นสำเนาเกือบเหมือนของแถวทดสอบ (การรั่วไหล, leakage; ดูหัวข้อ C)

In [ ]:
n_train = int(round(0.7 * len(df_w)))
X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]
print("train rows:", len(X_train), " test rows:", len(X_test))

In [ ]:
print("same as the ready-made split column:", (ready["split"].iloc[:n_train] == "train").all()
      and (ready["split"].iloc[n_train:] == "test").all())

In [ ]:
plt.plot(y_train.index, y_train, label="train (early)")
plt.plot(y_test.index, y_test, lw=2.5, label="test (late)")
plt.axvline(y_test.index[0], color="gray", ls="--")
plt.title(f"Chronological split of {SELECTED_TIME_SERIES_SIGNAL}"); plt.xlabel("time index t")
plt.ylabel("value"); plt.legend(); plt.show()

## Step 8 · Create and train the model
`LinearRegression` learns one weight per input: `y_hat = b0 + b1*x1 + ... + b5*x5`. This is an **autoregressive (AR)** model: a signal predicted from its own past, like a difference equation whose coefficients come from data. We fit on the training rows only.

`LinearRegression` เรียนรู้น้ำหนักหนึ่งค่าต่ออินพุต: `y_hat = b0 + b1*x1 + ... + b5*x5` นี่คือโมเดล **ออโตรีเกรสซีฟ (autoregressive, AR)**: สัญญาณที่ถูกทำนายจากอดีตของตัวเอง เหมือนสมการผลต่าง (difference equation) ที่สัมประสิทธิ์มาจากข้อมูล เราฟิตกับแถวชุดฝึกเท่านั้น

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
eq = " + ".join(f"{b:.3f}*x{k}" for k, b in enumerate(model.coef_, 1))
print(f"y_hat = {model.intercept_:.3f} + {eq}")

In [ ]:
print("x5 is the newest sample, x1 the oldest.")

## Step 9 · Predict
**One step ahead**: at every time the model is given the *real* last 5 values and predicts only the next one, so errors do not pile up.

**ทำนายหนึ่งก้าวข้างหน้า**: ทุกช่วงเวลา โมเดลได้รับ 5 ค่าล่าสุดที่ *เป็นค่าจริง* และทำนายเพียงค่าถัดไป ความคลาดเคลื่อนจึงไม่สะสม

In [ ]:
y_pred = model.predict(X_test)
pd.DataFrame({"actual": y_test, "y_pred": y_pred}).round(2).head()

## Step 10 · Evaluate
MAE = average size of the error. RMSE = like MAE but punishes big misses. MAPE = error in percent. R² = share of the ups and downs explained (1 is perfect). The **persistence baseline** ("next value = last value", i.e. `x5`) is the trivial rule any useful model must beat.

MAE = ขนาดเฉลี่ยของความคลาดเคลื่อน RMSE = คล้าย MAE แต่ลงโทษความคลาดเคลื่อนที่ใหญ่ MAPE = ความคลาดเคลื่อนเป็นเปอร์เซ็นต์ R² = สัดส่วนการขึ้นลงที่อธิบายได้ (1 คือสมบูรณ์แบบ) **เส้นฐานแบบคงเดิม (persistence baseline)** ("ค่าถัดไป = ค่าล่าสุด" คือ `x5`) เป็นกฎพื้นฐานที่โมเดลที่มีประโยชน์ต้องทำได้ดีกว่า

In [ ]:
# three sets of predictions to score: label -> (actual, predicted)
y_pred_train = model.predict(X_train)
y_base = X_test["x5"]                      # persistence baseline: next value = last value
sets = {"train": (y_train, y_pred_train),
        "test": (y_test, y_pred),
        "test, persistence baseline": (y_test, y_base)}

for label in sets:
    y_true, y_hat = sets[label]
    print("=====", label, "=====")

    mae = mean_absolute_error(y_true, y_hat)
    print("MAE  =", round(mae, 3))

    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    print("RMSE =", round(rmse, 3))

    mape = 100 * mean_absolute_percentage_error(y_true, y_hat)
    print("MAPE =", round(mape, 2), "%")

    r2 = r2_score(y_true, y_hat)
    print("R2   =", round(r2, 3))
    print()

## Step 11 · Visualize y and y_pred
Left: actual and predicted over time (test region shaded). Right: actual vs predicted; points on the diagonal are perfect.

ซ้าย: ค่าจริงและค่าที่ทำนายตามเวลา (บริเวณทดสอบแรเงา) ขวา: ค่าจริงเทียบกับค่าที่ทำนาย จุดบนเส้นทแยงมุมคือการทำนายที่สมบูรณ์แบบ

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(y.index, y, color="gray", label="actual")
a1.plot(y_test.index, y_pred, color="tab:red", label="one-step y_pred")
a1.axvspan(y_test.index[0], y_test.index[-1], color="tab:blue", alpha=0.12, label="test")
a1.set_title(f"{SELECTED_TIME_SERIES_SIGNAL} over time"); a1.set_xlabel("t"); a1.legend()
a2.scatter(y_test, y_pred, s=14)
lim = [y_test.min(), y_test.max()]
a2.plot(lim, lim, "k--"); a2.set_title("Actual vs predicted (test)")
a2.set_xlabel("actual"); a2.set_ylabel("predicted")
plt.tight_layout(); plt.show()

## A · Multi-step (recursive) forecast
Now pretend there are **no new measurements** in the test period. Start from the last 5 known values, predict one step, feed the prediction back in as input, and repeat (dead reckoning). Errors can now build up.

ทีนี้สมมติว่า **ไม่มีการวัดใหม่** ในช่วงทดสอบ เริ่มจาก 5 ค่าล่าสุดที่รู้ ทำนายหนึ่งก้าว แล้วป้อนค่าที่ทำนายกลับเป็นอินพุต และทำซ้ำ (เหมือนการคำนวณตำแหน่งโดยประมาณ, dead reckoning) ความคลาดเคลื่อนจึงสะสมได้

In [ ]:
window = list(X_test.iloc[0])               # last 5 real values before the test period
rec = []
for _ in range(len(X_test)):
    nxt = model.predict(pd.DataFrame([window], columns=feats))[0]
    rec.append(nxt)
    window = window[1:] + [nxt]               # drop oldest, add the prediction
rec = np.array(rec)

# compare the two kinds of prediction: label -> predicted values
preds = {"one-step": y_pred, "recursive": rec}

for label in preds:
    y_hat = preds[label]
    print("=====", label, "=====")

    mae = mean_absolute_error(y_test, y_hat)
    print("MAE  =", round(mae, 3))

    rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    print("RMSE =", round(rmse, 3))

    mape = 100 * mean_absolute_percentage_error(y_test, y_hat)
    print("MAPE =", round(mape, 2), "%")

    r2 = r2_score(y_test, y_hat)
    print("R2   =", round(r2, 3))
    print()

In [ ]:
plt.plot(y_train.index, y_train, color="lightgray", label="train")
plt.plot(y_test.index, y_test, color="black", lw=2, label="actual")
plt.plot(y_test.index, y_pred, color="tab:red", label="one-step")
plt.plot(y_test.index, rec, color="tab:blue", ls="--", lw=2, label="recursive")
plt.title("Actual vs one-step vs recursive"); plt.xlabel("t"); plt.legend(); plt.show()

The one-step line hugs the actual signal; the recursive line drifts away because it never sees real data again.

เส้นแบบหนึ่งก้าวแนบสัญญาณจริงอย่างใกล้ชิด ส่วนเส้นแบบเรียกซ้ำ (recursive) ค่อย ๆ เบี่ยงออกไป เพราะไม่เคยเห็นข้อมูลจริงอีกเลย

## B · Window size
Is a bigger window always better? Try `w = 1..10` on `trend_season` and `sine` and compare the one-step test MAE (chronological split).

หน้าต่างใหญ่กว่าดีกว่าเสมอหรือไม่? ลอง `w = 1..10` กับ `trend_season` และ `sine` แล้วเปรียบเทียบ MAE บนชุดทดสอบแบบหนึ่งก้าว (แบ่งตามลำดับเวลา)

In [ ]:
signals = [SELECTED_TIME_SERIES_SIGNAL, "sine"]
windows = range(1, 11)
mae_by_signal = {}                          # signal name -> list of test MAE, one per w

for name in signals:
    series = df[name]
    maes = []
    for ww in windows:
        # build the window table for this w
        d = pd.DataFrame({f"x{k}": series.shift(ww - k + 1) for k in range(1, ww + 1)})
        d["y"] = series
        d = d.dropna()
        # chronological 70/30 split
        k_train = int(round(0.7 * len(d)))
        X_w = d.drop(columns="y")
        y_w = d["y"]
        # train, predict, score
        m = LinearRegression()
        m.fit(X_w.iloc[:k_train], y_w.iloc[:k_train])
        pred_w = m.predict(X_w.iloc[k_train:])
        maes.append(mean_absolute_error(y_w.iloc[k_train:], pred_w))
    mae_by_signal[name] = maes

In [ ]:
piv = pd.DataFrame(mae_by_signal, index=pd.Index(windows, name="w"))
display(piv.round(3))

In [ ]:
piv.plot(marker="o"); plt.title("Test MAE vs window size w")
plt.ylabel("test MAE"); plt.show()

A sine wave is fully described by 2 lags (position and direction), so most of the gain comes early; the rest is noise. For `trend_season` the error does not fall steadily, so more lags are not automatically better.

คลื่นไซน์อธิบายได้ครบด้วย 2 lag (ตำแหน่งและทิศทาง) ดังนั้นการปรับปรุงส่วนใหญ่เกิดขึ้นตั้งแต่ช่วงต้น ส่วนที่เหลือคือ noise สำหรับ `trend_season` ความคลาดเคลื่อนไม่ลดลงอย่างสม่ำเสมอ ดังนั้น lag มากขึ้นไม่ได้ดีขึ้นโดยอัตโนมัติ

## C · Random vs chronological split
If we shuffle the rows, each test row is almost a copy of a training row, so the error can look better than it honestly is. We compare both splits (random split averaged over 20 shuffles) with `w = 5`.

ถ้าสับแถว แถวทดสอบแต่ละแถวจะเกือบเป็นสำเนาของแถวฝึก ความคลาดเคลื่อนจึงดูดีกว่าความจริง เราเปรียบเทียบการแบ่งทั้งสองแบบ (การแบ่งแบบสุ่มเฉลี่ยจากการสับ 20 ครั้ง) โดยใช้ `w = 5`

In [ ]:
signals = [SELECTED_TIME_SERIES_SIGNAL, "ar1", "random_walk", "sine"]
w5 = 5
leak_rows = {}                              # signal name -> {"chronological MAE", "random MAE"}

for name in signals:
    series = df[name]
    # build the window table (w = 5)
    d = pd.DataFrame({f"x{k}": series.shift(w5 - k + 1) for k in range(1, w5 + 1)})
    d["y"] = series
    d = d.dropna()
    X_w = d.drop(columns="y")
    y_w = d["y"]

    # chronological split: early 70 % train, late 30 % test
    k_train = int(round(0.7 * len(d)))
    m = LinearRegression()
    m.fit(X_w.iloc[:k_train], y_w.iloc[:k_train])
    chrono = mean_absolute_error(y_w.iloc[k_train:], m.predict(X_w.iloc[k_train:]))

    # random split, repeated with 20 different shuffles, then averaged
    rnd_list = []
    for seed in range(20):
        Xa, Xb, ya, yb = train_test_split(X_w, y_w, test_size=0.3, random_state=seed)
        m = LinearRegression()
        m.fit(Xa, ya)
        rnd_list.append(mean_absolute_error(yb, m.predict(Xb)))

    leak_rows[name] = {"chronological MAE": chrono, "random MAE": np.mean(rnd_list)}

In [ ]:
leak = pd.DataFrame(leak_rows).T
display(leak.round(3))

In [ ]:
leak.plot.bar(rot=0); plt.title("Test MAE: chronological vs random split"); plt.ylabel("test MAE"); plt.show()

The random split reports a different (often smaller) error than the honest chronological one. A simple linear model leaks only a little; flexible models suffer far more. Rule: for forecasting, always split by time.

การแบ่งแบบสุ่มรายงานความคลาดเคลื่อนที่ต่างออกไป (มักเล็กกว่า) การแบ่งตามเวลาที่ซื่อสัตย์ โมเดลเชิงเส้นอย่างง่ายรั่วไหลเพียงเล็กน้อย แต่โมเดลที่ยืดหยุ่นได้รับผลหนักกว่ามาก กฎคือ: การพยากรณ์ต้องแบ่งตามเวลาเสมอ

## Summary
- A time series is **one array** in time order; a **sliding window** (`shift`) turns it into a table with `n+1-w` rows, and a linear model on lags is an **AR(w)** model.
- **Split chronologically** (early = train, late = test); random splits leak the future.
- **One-step** predictions use real data; **recursive** forecasts reuse their own predictions, so error grows with the horizon.
- Always compare with the **persistence baseline**; for a random walk it is hard to beat. Bigger `w` is not always better.

- อนุกรมเวลาคือ **อาร์เรย์เดียว** เรียงตามเวลา **หน้าต่างเลื่อน** (`shift`) แปลงเป็นตารางที่มี `n+1-w` แถว และโมเดลเชิงเส้นบน lag คือโมเดล **AR(w)**
- **แบ่งตามลำดับเวลา** (ช่วงแรก = ฝึก, ช่วงหลัง = ทดสอบ) การแบ่งแบบสุ่มทำให้อนาคตรั่วไหล
- การทำนาย **หนึ่งก้าว** ใช้ข้อมูลจริง ส่วนการพยากรณ์ **แบบเรียกซ้ำ** นำค่าทำนายของตัวเองกลับมาใช้ ความคลาดเคลื่อนจึงเพิ่มขึ้นตามระยะพยากรณ์
- เปรียบเทียบกับ **persistence baseline** เสมอ สำหรับ random walk เส้นฐานนี้เอาชนะได้ยาก `w` ที่ใหญ่กว่าไม่ได้ดีกว่าเสมอไป

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---